# Latihan 1

In [13]:
!pip install faker

In [14]:
import numpy as np
import pandas as pd
from faker import Faker
import random

In [15]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} ",
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Kode ini menjalankan ulang seluruh proses pembuatan dataset dengan SEED diubah dari 42 menjadi 7, sehingga data acak yang dihasilkan akan berbeda dari sebelumnya. Jumlah baris transaksi_mentah.csv tetap sama yaitu 515 baris (500 asli + 15 duplikat) karena struktur pembuatan datanya sama, namun isi datanya (nama, produk, harga, tanggal) akan berbeda karena SEED yang berbeda menghasilkan urutan nilai acak yang berbeda. Setelah proses cleaning nanti, jumlah baris transaksi_bersih.csv kemungkinan besar akan berbeda dengan hasil SEED 42 karena baris yang terkena missing value dan duplikasi berada di posisi yang berbeda.

In [16]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


Kode ini mendeteksi jumlah data kosong di setiap kolom, hasilnya ada 20 nama pelanggan, 16 metode pembayaran, 30 kota, dan 166 rating yang kosong. Baris dengan nama pelanggan atau metode pembayaran kosong dihapus karena informasi ini wajib ada, sedangkan kota yang kosong diisi dengan teks "Tidak Diketahui" agar datanya tetap berguna untuk analisis lain.

In [17]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

In [18]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


Kode ini mendeteksi baris yang benar-benar identik (duplicate) dan menemukan 5 baris duplikat. Setelah baris duplikat dihapus menggunakan drop_duplicates(), jumlah data berkurang dari 515 menjadi 490 baris, sehingga hanya menyisakan data transaksi yang unik saja.

In [19]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

In [20]:
# a. Standardisasi teks kategorikal (category, payment_method, shipping_city):

for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" adalah singkatan; kembalikan ke huruf kapital penuh setelah Title Case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})


# b. Koreksi tipe data pada kolom price (dari teks bercampur simbol, menjadi numerik):

def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)


# c. Standardisasi format tanggal ke YYYY-MM-DD:

def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")


# d. Finalisasi tipe data:

df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Kode ini merapikan seluruh format data agar konsisten dan siap dianalisis dengan empat langkah utama. Pertama, menyeragamkan teks kategorikal menjadi Title Case dan menghapus spasi berlebih, khusus "COD" dikembalikan ke huruf kapital semua. Kedua, membersihkan kolom harga dari simbol "Rp", titik ribuan, dan spasi lalu mengubahnya menjadi angka murni (float). Ketiga, menyeragamkan format tanggal yang berantakan (ISO, DD/MM/YYYY, DD-MM-YYYY) menjadi satu format standar YYYY-MM-DD dengan mencoba berbagai format secara berurutan hingga berhasil.

# Latihan 2

In [21]:
# Tambahkan kolom is_valid_price
df["is_valid_price"] = df["price"] > 0

# Periksa apakah ada harga tidak valid
invalid_count = (~df["is_valid_price"]).sum()
print(f"Jumlah harga tidak valid (price <= 0 atau NaN): {invalid_count}")

if invalid_count > 0:
    print("Detail harga tidak valid:")
    print(df[~df["is_valid_price"]][["transaction_id", "product_name", "price", "is_valid_price"]])

Jumlah harga tidak valid (price <= 0 atau NaN): 0


Kode tersebut membuat kolom baru bernama is_valid_price yang menandai setiap baris dengan True apabila harga bernilai positif (> 0), dan False apabila harga nol, negatif, atau kosong (NaN). Selanjutnya, kode menghitung berapa banyak transaksi yang memiliki harga tidak valid serta menampilkan rinciannya jika ada. Dengan demikian, kita dapat melacak transaksi bermasalah pada kolom harga untuk kemudian diperbaiki atau dihapus.

In [22]:
# K-6. Ekspor Dataset Bersih

df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Kode ini mengekspor dataset yang telah melalui proses pembersihan ke dalam file transaksi_bersih.csv tanpa menyertakan kolom indeks. Dari 515 baris data mentah awal, tersisa 490 baris setelah eliminasi entri kosong dan duplikat. Dataset ini kini siap dimanfaatkan untuk keperluan analisis atau praktikum selanjutnya.

# Latihan 3

In [23]:
# Hitung jumlah transaksi per category
print("\n=== Jumlah Transaksi per Category ===")
transaksi_per_kategori = df["category"].value_counts()
print(transaksi_per_kategori)

# Opsional: Tampilkan dalam bentuk persentase
print("\n=== Persentase Transaksi per Category ===")
persentase_per_kategori = df["category"].value_counts(normalize=True) * 100
print(persentase_per_kategori.round(2))


=== Jumlah Transaksi per Category ===
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64

=== Persentase Transaksi per Category ===
category
Rumah Tangga    18.57
Kesehatan       17.55
Buku            16.53
Fashion         16.33
Elektronik      15.92
Olahraga         15.1
Name: proportion, dtype: Float64


Kode ini ngitung berapa kali tiap kategori produk muncul di dataset pakai value_counts(), terus nampilin hasilnya dalam dua gaya. Gaya pertama nunjukin jumlah transaksi per kategori dalam angka biasa (contohnya Elektronik: 95 transaksi), sedangkan gaya kedua nampilin dalam bentuk persentase (misalnya Elektronik: 19.59%) biar kita bisa lebih gampang lihat seberapa besar porsi tiap kategori dalam keseluruhan distribusi transaksi.

In [25]:
# Upload file CSV ke Google Drive

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Buat folder khusus (jika belum ada)
folder_path = "/content/drive/MyDrive/BigData/Praktikum2"
os.makedirs(folder_path, exist_ok=True)

# Copy file ke folder tersebut
!cp transaksi_mentah.csv "{folder_path}/"
!cp transaksi_bersih.csv "{folder_path}/"

print("✅ File berhasil diupload ke Google Drive!")
print(f"📁 Lokasi: MyDrive/BigData/Praktikum2")
print(f"📄 File yang diupload:")
print(f"   - transaksi_mentah.csv")
print(f"   - transaksi_bersih.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ File berhasil diupload ke Google Drive!
📁 Lokasi: MyDrive/BigData/Praktikum2
📄 File yang diupload:
   - transaksi_mentah.csv
   - transaksi_bersih.csv
